# Dataset Understanding and Problem Framing

## Problem Statement

The objective of this project is to perform sentiment analysis on the IMDB movie reviews dataset. Given a movie review as text, the task is to classify it as expressing either positive or negative sentiment. The project further aims to compare multiple text vectorization techniques to analyze their impact on classification performance.

## Dataset Description

The IMDB dataset consists of 50,000 movie reviews labeled as either positive or negative. The dataset is balanced, making accuracy a suitable primary evaluation metric. Reviews vary in length and contain informal language, HTML tags, and punctuation, making it a suitable benchmark for natural language processing tasks.

# Importing Dependencies

In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Data Loading

## Load Dataset

In [2]:
df=pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


## Dataset Shape

In [5]:
df.shape

(50000, 2)

## 'Sentiment' category count

In [6]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

## Encode target variable

In [7]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0}).astype(int)

In [8]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [9]:
df['sentiment'].value_counts()

sentiment
1    25000
0    25000
Name: count, dtype: int64

# Train Test Split

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
train_data,test_data=train_test_split(df,test_size=0.2,random_state=42)

In [12]:
train_data.head()

,review,sentiment
39087,That's what I kept asking myself during the ma...,0
30893,I did not watch the entire movie. I could not ...,0
45278,A touching love story reminiscent of In the M...,1
16398,This latter-day Fulci schlocker is a totally a...,0
13653,"First of all, I firmly believe that Norwegian ...",0


In [13]:
test_data.head()

,review,sentiment
33553,I really liked this Summerslam due to the look...,1
9427,Not many television shows appeal to quite as m...,1
199,The film quickly gets to a major chase scene w...,0
12447,Jane Austen would definitely approve of this o...,1
39489,Expectations were somewhat high for me when I ...,0


In [14]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


# Data Preprocessing

## Tokenize Data

In [15]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['review'])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test  = pad_sequences(tokenizer.texts_to_sequences(test_data['review']),  maxlen=200)

In [16]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'a': 3,
 'of': 4,
 'to': 5,
 'is': 6,
 'br': 7,
 'in': 8,
 'it': 9,
 'i': 10,
 'this': 11,
 'that': 12,
 'was': 13,
 'as': 14,
 'for': 15,
 'with': 16,
 'movie': 17,
 'but': 18,
 'film': 19,
 'on': 20,
 'not': 21,
 'you': 22,
 'are': 23,
 'his': 24,
 'have': 25,
 'be': 26,
 'one': 27,
 'he': 28,
 'all': 29,
 'at': 30,
 'by': 31,
 'an': 32,
 'they': 33,
 'so': 34,
 'from': 35,
 'who': 36,
 'like': 37,
 'or': 38,
 'just': 39,
 'her': 40,
 'out': 41,
 'about': 42,
 'if': 43,
 "it's": 44,
 'has': 45,
 'there': 46,
 'some': 47,
 'what': 48,
 'good': 49,
 'more': 50,
 'very': 51,
 'when': 52,
 'up': 53,
 'no': 54,
 'time': 55,
 'my': 56,
 'even': 57,
 'she': 58,
 'would': 59,
 'which': 60,
 'only': 61,
 'story': 62,
 'really': 63,
 'see': 64,
 'their': 65,
 'had': 66,
 'can': 67,
 'me': 68,
 'well': 69,
 'were': 70,
 'we': 71,
 'than': 72,
 'much': 73,
 'bad': 74,
 'been': 75,
 'do': 76,
 'get': 77,
 'great': 78,
 'also': 79,
 'will': 80,
 'other': 81,
 'into': 82,
 'p

In [17]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [18]:
print(X_test)

[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


## Separate input and output

In [19]:
y_train=train_data['sentiment']
y_test=test_data['sentiment']

In [20]:
print(y_train)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


In [21]:
print(y_test)

33553    1
9427     1
199      0
12447    1
39489    0
        ..
28567    0
25079    1
18707    1
15200    0
5857     1
Name: sentiment, Length: 10000, dtype: int64


# Model Building

In [22]:
model = Sequential()
model.add(Input(shape=(200,)))
model.add(Embedding(input_dim=5000, output_dim=128))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 200, 128)            │         640,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

# Compile Model

In [24]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

# Model Training

In [25]:
model.fit(X_train,y_train, epochs=5,batch_size=64,validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 356s 701ms/step - accuracy: 0.7844 - loss: 0.4634 - val_accuracy: 0.8487 - val_loss: 0.3594
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 337s 673ms/step - accuracy: 0.8616 - loss: 0.3385 - val_accuracy: 0.8309 - val_loss: 0.3741
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 369s 739ms/step - accuracy: 0.8778 - loss: 0.3018 - val_accuracy: 0.8680 - val_loss: 0.3452
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 359s 719ms/step - accuracy: 0.8669 - loss: 0.3187 - val_accuracy: 0.8729 - val_loss: 0.3206
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 330s 661ms/step - accuracy: 0.8910 - loss: 0.2720 - val_accuracy: 0.8767 - val_loss: 0.3138


# Model Evaluation

In [26]:
loss,accuracy=model.evaluate(X_test,y_test)
print(loss)
print(accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 64ms/step - accuracy: 0.8781 - loss: 0.3118
0.3118009865283966
0.8780999779701233
